In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = '/content/drive/MyDrive'

ROOT = None
for root, dirs, files in os.walk(BASE):
    if root.count('/') - BASE.count('/') > 3:
        dirs[:] = []
        continue
    for d in dirs:
        if 'labequip' in d.lower():
            ROOT = os.path.join(root, d)
            break
    if ROOT:
        break

print('ROOT:', ROOT)
print(os.listdir(ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT: /content/drive/MyDrive/LabEquipVis An Image Dataset of Computer Laborator
['LabEquipVis An Image Dataset of Computer Laborator']


In [2]:
import random
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

device: cuda


In [3]:
def sub(parent, key):
    for name in os.listdir(parent):
        if name.lower() == key:
            return os.path.join(parent, name)
    return None

ORIGINAL = ROOT # Initialize ORIGINAL with ROOT

# images + labels duitai ache emon sob folder khuje ber koro
pairs = []
for cur, dirs, files in os.walk(ORIGINAL):
    low = [d.lower() for d in dirs]
    if 'images' in low and 'labels' in low:
        pairs.append(cur)

print('found', len(pairs), 'image/label folders:')
for p in pairs:
    print(' ', p)

# proti image er jonno (image path, label path, split)
items = []
for p in pairs:
    img_dir = sub(p, 'images')
    lbl_dir = sub(p, 'labels')

    low = p.lower()
    if 'train' in low:
        split = 'train'
    elif 'val' in low:
        split = 'val'
    elif 'test' in low:
        split = 'test'
    else:
        split = None

    for fname in os.listdir(img_dir):
        name = os.path.splitext(fname)[0]
        lbl = os.path.join(lbl_dir, name + '.txt')
        if os.path.exists(lbl):
            items.append([os.path.join(img_dir, fname), lbl, split])

print('total images:', len(items))

# split folder na thakle nijera 70/20/10 kore nebo
if all(x[2] is None for x in items):
    random.shuffle(items)
    n = len(items)
    for i in range(n):
        if i < int(0.7 * n):
            items[i][2] = 'train'
        elif i < int(0.9 * n):
            items[i][2] = 'val'
        else:
            items[i][2] = 'test'
    print('own 70/20/10 split made')

for s in ['train', 'val', 'test']:
    print(s, ':', sum(1 for x in items if x[2] == s))

found 6 image/label folders:
  /content/drive/MyDrive/LabEquipVis An Image Dataset of Computer Laborator/LabEquipVis An Image Dataset of Computer Laborator/Original Data/train
  /content/drive/MyDrive/LabEquipVis An Image Dataset of Computer Laborator/LabEquipVis An Image Dataset of Computer Laborator/Original Data/valid
  /content/drive/MyDrive/LabEquipVis An Image Dataset of Computer Laborator/LabEquipVis An Image Dataset of Computer Laborator/Original Data/test
  /content/drive/MyDrive/LabEquipVis An Image Dataset of Computer Laborator/LabEquipVis An Image Dataset of Computer Laborator/Augmented Data/valid
  /content/drive/MyDrive/LabEquipVis An Image Dataset of Computer Laborator/LabEquipVis An Image Dataset of Computer Laborator/Augmented Data/train
  /content/drive/MyDrive/LabEquipVis An Image Dataset of Computer Laborator/LabEquipVis An Image Dataset of Computer Laborator/Augmented Data/test
total images: 10336
train : 7236
val : 2064
test : 1036


In [4]:
CLASS_NAMES = ['AC', 'Chair', 'CPU', 'Digital_Board', 'Fire_Extinguisher',
               'Keyboard', 'Light', 'Monitor', 'Mouse', 'Projector']

ids = set()
for img_path, lbl_path, split in items:
    for line in open(lbl_path):
        if line.strip():
            ids.add(int(line.split()[0]))

print('class ids found:', sorted(ids))
print('names given    :', len(CLASS_NAMES))

KeyboardInterrupt: 

In [ ]:
IMG_SIZE = 128
OUT = '/content/lab_cls'

count = 0
for img_path, lbl_path, split in items:
    img = Image.open(img_path).convert('RGB')
    W, H = img.size
    name = os.path.splitext(os.path.basename(img_path))[0]

    k = 0
    for line in open(lbl_path):
        parts = line.split()
        if len(parts) < 5:
            continue

        cid = int(parts[0])
        cx = float(parts[1]) * W
        cy = float(parts[2]) * H
        bw = float(parts[3]) * W
        bh = float(parts[4]) * H

        x1 = max(0, int(cx - bw / 2))
        y1 = max(0, int(cy - bh / 2))
        x2 = min(W, int(cx + bw / 2))
        y2 = min(H, int(cy + bh / 2))

        if x2 - x1 < 40 or y2 - y1 < 40:
            continue

        if cid < len(CLASS_NAMES):
            cname = CLASS_NAMES[cid]
        else:
            cname = 'class_' + str(cid)

        folder = os.path.join(OUT, split, cname)
        os.makedirs(folder, exist_ok=True)

        crop = img.crop((x1, y1, x2, y2)).resize((IMG_SIZE, IMG_SIZE))
        crop.save(os.path.join(folder, name + '_' + str(k) + '.jpg'))
        k += 1
        count += 1

print('total crops saved:', count)

In [ ]:
# How many crops per class in each split

for split in ['train', 'val', 'test']:
    print('-----', split)
    total = 0
    for c in sorted(os.listdir(os.path.join(OUT, split))):
        n = len(os.listdir(os.path.join(OUT, split, c)))
        total += n
        print(c, ':', n)
    print('total :', total)
    print()

In [ ]:
# ROUND 1 data pipeline: resize + ToTensor only, no augmentation

BATCH_SIZE = 32

basic_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

train_data = datasets.ImageFolder(OUT + '/train', transform=basic_transform)
val_data = datasets.ImageFolder(OUT + '/val', transform=basic_transform)
test_data = datasets.ImageFolder(OUT + '/test', transform=basic_transform)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

class_names = train_data.classes
num_classes = len(class_names)

print('classes:', class_names)
print('train:', len(train_data), ' val:', len(val_data), ' test:', len(test_data))



In [ ]:
# Look at a few training images

images, labels = next(iter(train_loader))

plt.figure(figsize=(12, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(images[i].permute(1, 2, 0))
    plt.title(class_names[labels[i]])
    plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# The CNN: 3 conv blocks + a small classification head

class SimpleCNN(nn.Module):
    def __init__(self, num_classes, p_drop=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        self.dropout = nn.Dropout(p_drop)
        self.fc1 = nn.Linear(64 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))    # 128 -> 64
        x = self.pool(F.relu(self.conv2(x)))    # 64  -> 32
        x = self.pool(F.relu(self.conv3(x)))    # 32  -> 16
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model1 = SimpleCNN(num_classes, p_drop=0.0).to(device)
print(model1)
print('parameters:', sum(p.numel() for p in model1.parameters()))



In [ ]:
# Training and validation loops

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total


def validate_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total


def run_training(model, tr_loader, va_loader, epochs):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(epochs):
        tr_loss, tr_acc = train_epoch(model, tr_loader, optimizer, criterion)
        va_loss, va_acc = validate_epoch(model, va_loader, criterion)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(va_acc)

        print('Epoch', epoch + 1, '/', epochs,
              '| train_loss %.4f' % tr_loss, 'train_acc %.4f' % tr_acc,
              '| val_loss %.4f' % va_loss, 'val_acc %.4f' % va_acc)

    return history



In [ ]:
# ROUND 1: train without augmentation and without dropout

EPOCHS = 20
history1 = run_training(model1, train_loader, val_loader, EPOCHS)



In [ ]:
# Curves for Round 1

def plot_history(history, title):
    epochs = range(1, len(history['train_loss']) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'o-', label='train')
    plt.plot(epochs, history['val_loss'], 'o-', label='validation')
    plt.xlabel('epoch')
    plt.ylabel('loss')
    plt.title('Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], 'o-', label='train')
    plt.plot(epochs, history['val_acc'], 'o-', label='validation')
    plt.xlabel('epoch')
    plt.ylabel('accuracy')
    plt.title('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_history(history1, 'Round 1 - no augmentation, no dropout')

gap1 = history1['train_acc'][-1] - history1['val_acc'][-1]
print('final train acc: %.4f' % history1['train_acc'][-1])
print('final val acc  : %.4f' % history1['val_acc'][-1])
print('gap            : %.4f' % gap1)


# ===================== MARKDOWN CELL (paste as Text cell) =====================
"""
## Round 1 - overfitting

Training accuracy keeps climbing towards 1.0 and training loss keeps falling,
but validation accuracy flattens and validation loss stops falling and starts to
rise again. The growing distance between the two curves is overfitting: the
network is memorising the exact crops it was shown instead of learning what a
monitor or a keyboard generally looks like.

This is expected here. Many crops come from the same few laboratory rooms, so
the same equipment appears again and again with the same background and lighting,
which is very easy to memorise.
"""



In [ ]:
# ROUND 2 data pipeline: simple transformations on the training set only

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor()
])

train_data_aug = datasets.ImageFolder(OUT + '/train', transform=train_transform)
train_loader_aug = DataLoader(train_data_aug, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

print('augmented training set ready:', len(train_data_aug))
print('validation and test stay unchanged (no augmentation)')



In [ ]:
# The same image looks a little different every time it is loaded

plt.figure(figsize=(12, 3))
for i in range(6):
    img, lab = train_data_aug[0]
    plt.subplot(1, 6, i + 1)
    plt.imshow(img.permute(1, 2, 0))
    plt.title(class_names[lab])
    plt.axis('off')
plt.suptitle('Same image, 6 random transformations')
plt.tight_layout()
plt.show()

In [ ]:
# ROUND 2: same CNN, now with dropout, trained on the transformed data

model2 = SimpleCNN(num_classes, p_drop=0.5).to(device)
history2 = run_training(model2, train_loader_aug, val_loader, EPOCHS)


# ===================== CELL 16 =====================
plot_history(history2, 'Round 2 - transformations + dropout')

gap2 = history2['train_acc'][-1] - history2['val_acc'][-1]
print('final train acc: %.4f' % history2['train_acc'][-1])
print('final val acc  : %.4f' % history2['val_acc'][-1])
print('gap            : %.4f' % gap2)



In [ ]:
# Compare the two rounds

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history1['val_acc'], 'o-', label='Round 1')
plt.plot(history2['val_acc'], 'o-', label='Round 2')
plt.xlabel('epoch')
plt.ylabel('validation accuracy')
plt.title('Validation accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
gaps1 = [a - b for a, b in zip(history1['train_acc'], history1['val_acc'])]
gaps2 = [a - b for a, b in zip(history2['train_acc'], history2['val_acc'])]
plt.plot(gaps1, 'o-', label='Round 1')
plt.plot(gaps2, 'o-', label='Round 2')
plt.xlabel('epoch')
plt.ylabel('train acc - val acc')
plt.title('Overfitting gap')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print('Round 1 -> best val acc %.4f | final gap %.4f' % (max(history1['val_acc']), gap1))
print('Round 2 -> best val acc %.4f | final gap %.4f' % (max(history2['val_acc']), gap2))



In [ ]:
# Final check on the untouched test set

def predict_all(model, loader):
    model.eval()
    y_true = []
    y_pred = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            y_pred.extend(outputs.argmax(1).cpu().numpy())
            y_true.extend(labels.numpy())
    return np.array(y_true), np.array(y_pred)

true1, pred1 = predict_all(model1, test_loader)
true2, pred2 = predict_all(model2, test_loader)

print('Round 1 test accuracy: %.4f' % (true1 == pred1).mean())
print('Round 2 test accuracy: %.4f' % (true2 == pred2).mean())



In [ ]:
# Confusion matrix and per class report for the final model

import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(true2, pred2)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion matrix - test set (Round 2)')
plt.show()

print(classification_report(true2, pred2, target_names=class_names))



In [ ]:
# Some wrong predictions of the final model

model2.eval()
wrong = []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model2(images.to(device))
        preds = outputs.argmax(1).cpu()
        for img, t, p in zip(images, labels, preds):
            if t != p:
                wrong.append((img, t.item(), p.item()))

print('misclassified test crops:', len(wrong))

show = min(12, len(wrong))
plt.figure(figsize=(12, 7))
for i in range(show):
    img, t, p = wrong[i]
    plt.subplot(3, 4, i + 1)
    plt.imshow(img.permute(1, 2, 0))
    plt.title('true ' + class_names[t] + '\npred ' + class_names[p], fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()



In [ ]:
# What the first convolution layer learned

weights = model2.conv1.weight.data.cpu().clone()
weights = (weights - weights.min()) / (weights.max() - weights.min())

plt.figure(figsize=(10, 3))
for i in range(16):
    plt.subplot(2, 8, i + 1)
    plt.imshow(weights[i].permute(1, 2, 0))
    plt.axis('off')
plt.suptitle('16 filters of conv1')
plt.tight_layout()
plt.show()

img, lab = test_data[0]
x = img.unsqueeze(0).to(device)

model2.eval()
with torch.no_grad():
    maps = F.relu(model2.conv1(x)).cpu()[0]

plt.figure(figsize=(12, 5))
plt.subplot(2, 7, 1)
plt.imshow(img.permute(1, 2, 0))
plt.title(class_names[lab])
plt.axis('off')
for i in range(12):
    plt.subplot(2, 7, i + 2)
    plt.imshow(maps[i], cmap='viridis')
    plt.axis('off')
plt.suptitle('Feature maps from conv1')
plt.tight_layout()
plt.show()

